# SAR → structure → optical color: MRSHaze training

This notebook **clones your GitHub repository** and runs all MRSHaze training in Kaggle. First push the contents of this project to your repository; update `REPO_URL` below if needed.

In Kaggle Settings: enable a GPU and Internet, and attach your MRSHaze dataset. No dataset copying or local training is needed. The default URL assumes `sumire25/grayToColor`; it is configurable.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Update this if the new repository has a different owner/name.
REPO_URL = "https://github.com/sumire25/SARguidedRSID.git"
REPO_REF = None  # Optional branch/tag. Leave None for the repository default.
REPO_DIR = Path("/kaggle/working/SARguidedRSID")
DATA_ROOT = Path("/kaggle/input/datasets/markosumire/mrshaze/MRSHaze")

if not (REPO_DIR / ".git").is_dir():
    command = ["git", "clone", "--depth", "1"]
    if REPO_REF:
        command += ["--branch", REPO_REF]
    subprocess.run(command + [REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Source revision:", revision)
assert DATA_ROOT.is_dir(), f"Mount MRSHaze and update DATA_ROOT: {DATA_ROOT}"

import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU in Kaggle Settings before full training/testing."
print("GPU:", torch.cuda.get_device_name(0))


## Configure or resume

Default: **six E2E epochs**. Both stages learn jointly from the first batch, with grayscale supervision retained. There is no separate grayscale/color warm-up and no ground-truth grayscale teacher in this mode. The learning rate still has a one-epoch warm-up.

Validation is a fixed 10% holdout of **train** (5,400 optimization / 600 validation images); the 2,000 test images are unused here. Training uses synchronized random crops, rotations, flips, and 10% SAR dropout; validation uses complete images with none of these augmentations.

`MAX_TRAIN_HOURS=8.0` limits each invocation of the training script, including setup inside that script and validation. It leaves time for cloning, dependency installation, output finalization, and variation in Kaggle resources. Clock checks between batches/images have negligible overhead compared with model computation. This is a cooperative guard: it cannot interrupt a stuck operation, guarantee artifact retention after a crash, or guarantee all six epochs finish.

For a later session, attach the previous notebook output and set `RESUME` to its `last.pt`. Resume uses the **saved mode, phase lengths, optimizer settings, and split**; new-run controls below are ignored on resume, except the per-session time budget and mount/output paths. To switch an old hybrid checkpoint to E2E, use `INIT_WEIGHTS` with `RESUME=None` to start a new experiment and schedule.

For a hybrid or modular comparison, change `MODE` and explicitly choose nonzero `GRAY_EPOCHS` and `COLOR_EPOCHS` (for example 2/1/3). See [TRAINING.md](https://github.com/sumire25/grayToColor/blob/main/TRAINING.md) in your repository for the full protocol.


In [ ]:
import yaml
from graytocolor.runtime import load_checkpoint

RUN_DIR = Path("/kaggle/working/runs/sar_gray_color")
RESUME = None  # Example: "/kaggle/input/notebooks/markosumire/NOTEBOOK/runs/sar_gray_color/last.pt"
INIT_WEIGHTS = None  # Optional model weights for a NEW mode/schedule; never combine with RESUME.
STOP_AFTER_EPOCH = None  # Optional absolute completed-epoch count; e.g. 3.
MAX_TRAIN_HOURS = 11.0  # Each session gets a fresh budget. None disables the time guard.

MODE = "e2e"
GRAY_EPOCHS = 0
COLOR_EPOCHS = 0
JOINT_EPOCHS = 50
BATCH_SIZE = 10
CROP_SIZE = 256
ACCUMULATION_STEPS = 1

if RESUME:
    config = load_checkpoint(RESUME)["config"]
    print("Resuming saved mode/schedule:", config["mode"], {
        key: config["train"][key] for key in ("gray_epochs", "color_epochs", "joint_epochs")
    })
else:
    config = yaml.safe_load((REPO_DIR / "mrshaze.yaml").read_text())
    config["mode"] = MODE
    config["data"]["crop_size"] = CROP_SIZE
    config["train"].update(
        gray_epochs=GRAY_EPOCHS, color_epochs=COLOR_EPOCHS, joint_epochs=JOINT_EPOCHS,
        batch_size=BATCH_SIZE, accumulation_steps=ACCUMULATION_STEPS,
    )
    # Architecture/loss ablations can be changed here before a NEW run.
    # config["model"]["use_sar"] = False
    # config["model"]["use_structure_features"] = False

config.update(run_dir=str(RUN_DIR), device="cuda")
config["data"]["root"] = str(DATA_ROOT)
config["train"]["max_hours"] = MAX_TRAIN_HOURS
RUN_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = RUN_DIR / "kaggle_config.yaml"
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False))
(RUN_DIR / "source_revision.txt").write_text(revision + "\n")
print(CONFIG_PATH.read_text())


In [ ]:
command = [sys.executable, "-u", "train.py", "--config", str(CONFIG_PATH)]
if RESUME:
    command += ["--resume", str(RESUME)]
if INIT_WEIGHTS:
    command += ["--init-weights", str(INIT_WEIGHTS)]
if STOP_AFTER_EPOCH is not None:
    command += ["--stop-after-epoch", str(STOP_AFTER_EPOCH)]
subprocess.run(command, cwd=REPO_DIR, check=True)


## Inspect training artifacts and a possible budget stop

`last.pt` is written at initialization (epoch 0, untrained) and after every completed epoch **and its scheduled validation**. Logs explicitly confirm each epoch save. If the time guard stops during training or validation, the incomplete epoch is discarded and the previous completed checkpoint is retained. No mid-epoch continuation is claimed.

`best.pt` is chosen by validation RGB PSNR in the final configured phase. With E2E, selection starts after epoch 1. An epoch-zero checkpoint cannot be used for model evaluation. Warm-up best models are separate when those phases are configured. The script records an early budget stop in `session_stop.json` and returns normally so this notebook can finish.

Use **Save Version → Save & Run All** to run independently of the browser and preserve outputs on successful completion. Local checkpoint writes are not continuous cloud backups. Unexpected runtime termination/cancellation can prevent artifact publication. The guard cannot protect against all such failures.

When resuming into a new output folder, available earlier best checkpoints and metric history are carried forward from the mounted checkpoint directory. Keep the old output attached. The total schedule is unchanged; an eight-hour budget refreshes each session.


In [ ]:
import json
import matplotlib.pyplot as plt

stop_path = RUN_DIR / "session_stop.json"
if stop_path.exists():
    print("Time guard stopped training cleanly:")
    print(stop_path.read_text())

metrics_path = RUN_DIR / "metrics.jsonl"
records = [json.loads(line) for line in metrics_path.read_text().splitlines()] if metrics_path.exists() else []
if records:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot([r["epoch"] for r in records], [r["train"]["total"] for r in records])
    axes[0].set(xlabel="Epoch", ylabel="Training loss")
    for key, label in [("gray_psnr", "Recovered luminance"), ("psnr", "Dehazed RGB")]:
        selected = [r for r in records if key in r["val"]]
        if selected:
            axes[1].plot([r["epoch"] for r in selected], [r["val"][key] for r in selected], label=label)
    axes[1].set(xlabel="Epoch", ylabel="Validation PSNR (dB)")
    if axes[1].lines:
        axes[1].legend()
    plt.show()
else:
    print("No epoch has completed in the available history. last.pt is a restart checkpoint.")

print("Saved artifacts:")
for path in sorted(RUN_DIR.iterdir()):
    print(path.name)
